# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1: Install dependencies
# Run this cell first. Do NOT pin langchain to 0.3.x — Colab pre-installs
# langchain-classic / langgraph which require langchain-core >= 1.4.4.

!pip install -q \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-chroma \
  langchain-text-splitters \
  chromadb \
  pypdf \
  ipywidgets \
  sentence-transformers

# Enable ipywidgets rendering in Colab
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Print installed versions for verification
try:
    import importlib.metadata as _meta
except ImportError:
    import importlib_metadata as _meta

for _pkg in [
    "langchain", "langchain-core", "langchain-openai",
    "langchain-chroma", "langchain-text-splitters",
    "chromadb", "pypdf", "ipywidgets", "sentence-transformers",
]:
    try:
        print(f"{_pkg}=={_meta.version(_pkg)}")
    except Exception:
        print(f"{_pkg} not found")

print("\nInstallation complete. Run Cell 2 to launch the UI.")
print("You will enter your OpenAI API key directly in the UI.")

In [ ]:
# Colab Cell 2: RAG app with ipywidgets UI
# Run after Cell 1. Enter your OpenAI API key in the UI field below.

import io, re, html as _html, threading, uuid
from pypdf import PdfReader
from IPython.display import display
import ipywidgets as widgets

# Re-enable widget manager in case Cell 1 wasn't re-run
try:
    from google.colab import output as _colab_out
    _colab_out.enable_custom_widget_manager()
except Exception:
    pass

# Use the synchronous OpenAI client and chromadb directly —
# avoids the asyncio deadlock that occurs when langchain's
# async-backed ChatOpenAI is called from an ipywidgets thread.
import openai as _oai
import chromadb as _chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── State ─────────────────────────────────────────────────────────────────────

_state = {
    "chroma_client": None,   # chromadb.Client()
    "collection":    None,   # chromadb Collection
    "indexed_files": [],
}

# ── API helpers (all synchronous — safe to call from any thread) ──────────────

def get_api_key():
    return api_key_input.value.strip()

def _oai_client():
    return _oai.OpenAI(api_key=get_api_key())

def _embed(texts):
    """Batch-embed texts via the synchronous OpenAI API."""
    resp = _oai_client().embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in resp.data]

def _chat(prompt):
    """Single-turn chat completion via the synchronous OpenAI API."""
    resp = _oai_client().chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return resp.choices[0].message.content

# ── PDF + indexing ────────────────────────────────────────────────────────────

def pdf_bytes_to_text(b):
    reader = PdfReader(io.BytesIO(b))
    pages = []
    for p in reader.pages:
        try: pages.append(p.extract_text() or "")
        except: pages.append("")
    return "\n\n".join(pages)

_SPLITTER = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

def index_files(files):
    """
    files: list of (filename, bytes)
    Adds chunks to the shared in-memory chromadb collection.
    Returns the updated collection.
    """
    if _state["chroma_client"] is None:
        _state["chroma_client"] = _chromadb.Client()
        _state["collection"]    = _state["chroma_client"].create_collection("rag")

    col = _state["collection"]

    for fname, pdf_bytes in files:
        _set_status(f"Extracting text from '{fname}'…", "#aaa")
        text   = pdf_bytes_to_text(pdf_bytes)
        chunks = _SPLITTER.split_text(text)
        if not chunks:
            _set_status(f"No text found in '{fname}'. Skipped.", "orange")
            continue

        # Embed in batches of 100 (API limit is 2048 but 100 keeps it fast)
        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            batch     = chunks[i : i + batch_size]
            _set_status(f"Embedding chunks {i+1}–{i+len(batch)} / {len(chunks)} for '{fname}'…", "#aaa")
            embeddings = _embed(batch)
            ids        = [f"{fname}::{i+j}" for j in range(len(batch))]
            metas      = [{"source": fname}] * len(batch)
            col.add(documents=batch, embeddings=embeddings, ids=ids, metadatas=metas)

        if fname not in _state["indexed_files"]:
            _state["indexed_files"].append(fname)
        _refresh_indexed()

    return col

def retrieve(query, n=5):
    col = _state["collection"]
    if col is None:
        return []
    emb     = _embed([query])[0]
    results = col.query(query_embeddings=[emb], n_results=min(n, col.count()))
    return results["documents"][0] if results["documents"] else []

# ── Prompts ───────────────────────────────────────────────────────────────────

_NOT_FOUND = "I cannot find that information in the provided document."

_QA_TMPL = (
    "You are a helpful assistant. Answer ONLY from the CONTEXT below.\n"
    "Domain: {domain}\n\n"
    "Rules:\n"
    "- Use ONLY the CONTEXT. If the answer is absent, reply exactly: \"{not_found}\"\n"
    "- Be concise. Adapt style to the domain (Law → clauses; Telecom/Media → specs; General → neutral).\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n\nAnswer:"
)

_SUMMARY_TMPL = (
    "You are a summarization assistant. Summarize ONLY from the CONTEXT below.\n"
    "Domain: {domain}\n\n"
    "CONTEXT:\n{context}\n\n"
    "Produce a structured summary with clear headings. Note any missing key details briefly."
)

# ── RAG functions (purely synchronous) ───────────────────────────────────────

def query_with_rag(question, domain):
    _set_status("Step 1/2 — Retrieving relevant passages…", "#aaa")
    chunks  = retrieve(question)
    context = "\n\n".join(chunks) if chunks else ""

    _set_status("Step 2/2 — Generating answer…", "#aaa")
    prompt = _QA_TMPL.format(domain=domain, context=context,
                              question=question, not_found=_NOT_FOUND)
    answer = _chat(prompt)

    if _NOT_FOUND not in answer:
        return answer

    # Fallback — answer from general knowledge
    _set_status("Not in document — consulting general knowledge…", "#aaa")
    fb = _chat(
        f"Question: {question}\n\n"
        "The user's uploaded document did not contain relevant information. "
        "Please answer helpfully from your general knowledge."
    )
    return "⚠️ **Not found in uploaded document(s)** — answering from general knowledge:\n\n" + fb

def generate_summary(domain):
    _set_status("Step 1/2 — Retrieving key passages…", "#aaa")
    chunks  = retrieve("summary overview introduction conclusion key points", n=8)
    context = "\n\n".join(chunks) if chunks else ""
    _set_status("Step 2/2 — Generating summary…", "#aaa")
    return _chat(_SUMMARY_TMPL.format(domain=domain, context=context))

# ── Markdown → HTML helper ────────────────────────────────────────────────────

def _md_to_html(text):
    out = []
    for line in text.split("\n"):
        if line.startswith("### "):
            out.append(f"<h4 style='margin:8px 0 4px'>{_html.escape(line[4:])}</h4>")
        elif line.startswith("## "):
            out.append(f"<h3 style='margin:10px 0 4px'>{_html.escape(line[3:])}</h3>")
        elif line.startswith("# "):
            out.append(f"<h2 style='margin:12px 0 4px'>{_html.escape(line[2:])}</h2>")
        elif line.startswith(("- ", "* ")):
            out.append(f"<li style='margin:2px 0'>{_html.escape(line[2:])}</li>")
        elif line.strip() == "":
            out.append("<br>")
        else:
            s = _html.escape(line)
            s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)
            s = re.sub(r"\*(.+?)\*",     r"<i>\1</i>", s)
            out.append(f"<p style='margin:4px 0'>{s}</p>")
    return "".join(out)

# ── Widgets ───────────────────────────────────────────────────────────────────

# Step 1 — API key
api_key_input  = widgets.Password(value="", placeholder="sk-...",
                                   description="OpenAI Key:",
                                   layout=widgets.Layout(width="400px"))
api_key_status = widgets.HTML('<span style="color:orange">Enter your OpenAI API key</span>')

def _on_key_change(change):
    k = change["new"].strip()
    api_key_status.value = (
        '<span style="color:green">&#10003; Key set</span>'
        if k.startswith("sk-") and len(k) > 20
        else '<span style="color:orange">Enter a valid sk-... key</span>'
    )
api_key_input.observe(_on_key_change, names="value")

# Step 2 — Category
_DEFAULT_DOMAINS = ["Media", "Law", "Telecom", "General"]
domain_dd      = widgets.Dropdown(options=_DEFAULT_DOMAINS, value="General",
                                   description="Category:", layout=widgets.Layout(width="220px"))
new_cat_input  = widgets.Text(value="", placeholder="e.g. Finance, Healthcare…",
                               description="", layout=widgets.Layout(width="210px"))
add_cat_btn    = widgets.Button(description="+ Add",    button_style="warning", layout=widgets.Layout(width="70px"))
remove_cat_btn = widgets.Button(description="✕ Remove", button_style="danger",  layout=widgets.Layout(width="90px"))
cat_msg        = widgets.HTML("")

def on_add_cat(b):
    name = new_cat_input.value.strip()
    if not name: cat_msg.value = '<span style="color:orange">Type a name first.</span>'; return
    opts = list(domain_dd.options)
    if name in opts: cat_msg.value = f'<span style="color:orange">"{name}" already exists.</span>'; return
    opts.append(name); domain_dd.options = opts; domain_dd.value = name
    new_cat_input.value = ""
    cat_msg.value = f'<span style="color:green">&#10003; Added "{name}"</span>'
add_cat_btn.on_click(on_add_cat)

def on_remove_cat(b):
    cur = domain_dd.value
    if cur in _DEFAULT_DOMAINS: cat_msg.value = f'<span style="color:orange">Cannot remove built-in "{cur}".</span>'; return
    opts = [o for o in domain_dd.options if o != cur]
    domain_dd.options = opts; domain_dd.value = opts[-1] if opts else None
    cat_msg.value = f'<span style="color:green">&#10003; Removed "{cur}"</span>'
remove_cat_btn.on_click(on_remove_cat)

# Step 2 — Upload
upload_out         = widgets.Output(layout=widgets.Layout(border="1px dashed #555",
                                    padding="6px", min_height="30px", margin="4px 0"))
upload_btn         = widgets.Button(description="📂 Upload PDF(s)", button_style="info",
                                    layout=widgets.Layout(width="165px"))
clear_btn          = widgets.Button(description="Clear Index", button_style="danger",
                                    layout=widgets.Layout(width="110px"))
indexed_files_html = widgets.HTML('<i style="color:#888">No documents indexed yet</i>')

def _refresh_indexed():
    files = _state["indexed_files"]
    if not files:
        indexed_files_html.value = '<i style="color:#888">No documents indexed yet</i>'
        return
    rows = "".join(f'<li style="color:green;margin:2px 0">&#10003; {_html.escape(f)}</li>' for f in files)
    n = len(files)
    indexed_files_html.value = (
        f'<b>Indexed ({n} doc{"s" if n!=1 else ""}):</b>'
        f'<ul style="margin:4px 0;padding-left:18px">{rows}</ul>'
        f'<span style="font-size:12px;color:#aaa">Click "📂 Upload PDF(s)" again to add more.</span>'
    )

# Status + Answer (HTML widgets — thread-safe)
status_html = widgets.HTML('<i style="color:#888">Ready.</i>')
answer_html = widgets.HTML('<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>')

def _set_status(msg, color="#ccc"):
    status_html.value = f'<span style="color:{color}">{_html.escape(str(msg))}</span>'

def _set_answer(text):
    answer_html.value = (
        '<div style="font-family:sans-serif;font-size:14px;line-height:1.7;'
        'padding:10px;border:1px solid #555;border-radius:4px">'
        + _md_to_html(text) + '</div>'
    )

# Upload handler — calls files.upload() then indexes in a background thread
def on_upload_click(b):
    if not get_api_key():
        _set_status("Enter your OpenAI API key first.", "orange"); return
    upload_out.clear_output()
    try:
        from google.colab import files as _gf
    except ImportError:
        with upload_out: print("Not running in Colab."); return
    with upload_out:
        print("Opening file picker — select one or more PDFs…")
        try:
            raw = _gf.upload()
        except Exception as e:
            print("Upload error:", e); return
    upload_out.clear_output()
    if not raw:
        _set_status("No files uploaded.", "#aaa"); return
    pdf_files = [(fn, bytes(c)) for fn, c in raw.items() if fn.lower().endswith(".pdf")]
    skipped   = [fn for fn in raw if not fn.lower().endswith(".pdf")]
    if skipped: _set_status(f"Skipped non-PDF: {', '.join(skipped)}", "orange")
    if not pdf_files:
        _set_status("No valid PDF files — please upload .pdf files.", "orange"); return

    # Index in background thread (sync OpenAI client is thread-safe)
    def _worker():
        try:
            index_files(pdf_files)
            n = len(_state["indexed_files"])
            _set_status(f"✓ {n} doc(s) indexed. Ask a question or generate a summary.", "lightgreen")
        except Exception as e:
            _set_status(f"Indexing failed: {e}", "tomato")
    threading.Thread(target=_worker, daemon=True).start()
upload_btn.on_click(on_upload_click)

def on_clear_click(b):
    _state.update({"chroma_client": None, "collection": None, "indexed_files": []})
    _refresh_indexed()
    answer_html.value = '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
    _set_status("Index cleared. Upload new PDF(s) to start again.", "#aaa")
clear_btn.on_click(on_clear_click)

# Step 3 — Ask
ask_text = widgets.Text(value="", description="Question:", layout=widgets.Layout(width="70%"))
ask_btn  = widgets.Button(description="Ask", button_style="primary")

def on_ask(b):
    if not get_api_key():   _set_status("Enter your OpenAI API key first.", "orange"); return
    if not _state["collection"]: _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    q = ask_text.value.strip()
    if not q: _set_status("Please type a question.", "orange"); return
    domain = domain_dd.value
    ask_btn.disabled = True
    _set_status(f"Searching {len(_state['indexed_files'])} doc(s)…", "#aaa")
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Thinking…</i></div>'
    def _worker():
        try:
            ans = query_with_rag(q, domain)
            _set_answer(ans)
            _set_status("Done.", "lightgreen")
        except Exception as e:
            _set_status(f"Query failed: {e}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(e))}</div>'
        finally:
            ask_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
ask_btn.on_click(on_ask)

# Step 3 — Summary
summary_btn = widgets.Button(description="Generate Summary", button_style="info")

def on_summary(b):
    if not get_api_key():   _set_status("Enter your OpenAI API key first.", "orange"); return
    if not _state["collection"]: _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    domain = domain_dd.value
    summary_btn.disabled = True
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Generating summary…</i></div>'
    def _worker():
        try:
            summ = generate_summary(domain)
            _set_answer(summ)
            _set_status("Summary ready.", "lightgreen")
        except Exception as e:
            _set_status(f"Summary failed: {e}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(e))}</div>'
        finally:
            summary_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
summary_btn.on_click(on_summary)

# ── Layout ────────────────────────────────────────────────────────────────────

key_box = widgets.VBox([
    widgets.HTML("<b>Step 1 — Enter your OpenAI API Key</b>"),
    widgets.HBox([api_key_input, api_key_status]),
])
upload_box = widgets.VBox([
    widgets.HTML("<b>Step 2 — Choose Category &amp; Upload PDF(s)</b>"),
    domain_dd,
    widgets.HBox([new_cat_input, add_cat_btn, remove_cat_btn]),
    cat_msg,
    widgets.HTML("<div style='margin-top:6px'></div>"),
    widgets.HBox([upload_btn, clear_btn]),
    upload_out,
    indexed_files_html,
])
action_box = widgets.VBox([
    widgets.HTML("<b>Step 3 — Ask or Summarize</b>"),
    widgets.HTML("<span style='font-size:12px;color:#aaa'>Ask as many questions as you like. Upload more docs anytime.</span>"),
    widgets.HBox([ask_text, ask_btn]),
    widgets.HTML("<i style='margin:4px 0;display:block'>— or —</i>"),
    summary_btn,
])
ui = widgets.VBox([
    key_box,
    widgets.HTML("<hr>"),
    widgets.HBox([upload_box, action_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Status</b>"),
    status_html,
    widgets.HTML("<b style='display:block;margin-top:10px'>Answer</b>"),
    answer_html,
], layout=widgets.Layout(padding="10px"))

display(ui)